# NovaPay - Day 3: Data Cleaning & Preparation

This notebook cleans `data/raw/nova_pay_combined.csv` and saves the cleaned Day 3 dataset to `data/processed/cleaned_transactions.csv`.

Scope: cleaning only. This notebook does not include model training, train-test splitting, SMOTE, threshold tuning, or model evaluation.


## 1. Import libraries


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


## 2. Load the raw dataset


In [2]:
RAW_PATH = Path('data/raw/nova_pay_combined.csv')
PROCESSED_PATH = Path('data/processed/cleaned_transactions.csv')

df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()


## 3. Raw dataset overview


In [3]:
print(f'Raw shape: {df_raw.shape}')
display(df_raw.head())
print('Column names:')
print(df_raw.columns.tolist())
print('\nData types:')
display(df_raw.dtypes)
print('\nMissing values:')
display(df_raw.isna().sum().sort_values(ascending=False))
duplicate_count_before = int(df_raw.duplicated().sum())
print(f'Raw duplicate rows: {duplicate_count_before:,}')


Raw shape: (11400, 26)


,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


Column names:
['transaction_id', 'customer_id', 'timestamp', 'home_country', 'source_currency', 'dest_currency', 'channel', 'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest', 'device_id', 'new_device', 'ip_address', 'ip_country', 'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days', 'device_trust_score', 'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud']

Data types:


transaction_id                object
customer_id                   object
timestamp                     object
home_country                  object
source_currency               object
dest_currency                 object
channel                       object
amount_src                    object
amount_usd                   float64
fee                          float64
exchange_rate_src_to_dest    float64
device_id                     object
new_device                      bool
ip_address                    object
ip_country                    object
location_mismatch               bool
ip_risk_score                float64
kyc_tier                      object
account_age_days               int64
device_trust_score           float64
chargeback_history_count       int64
risk_score_internal          float64
txn_velocity_1h                int64
txn_velocity_24h               int64
corridor_risk                float64
is_fraud                       int64
dtype: object


Missing values:


amount_usd                   305
ip_address                   305
ip_country                   301
kyc_tier                     300
device_trust_score           295
fee                          295
timestamp                     29
transaction_id                 0
customer_id                    0
home_country                   0
channel                        0
amount_src                     0
source_currency                0
dest_currency                  0
new_device                     0
device_id                      0
location_mismatch              0
exchange_rate_src_to_dest      0
ip_risk_score                  0
account_age_days               0
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

Raw duplicate rows: 200


## 4. Preserve label integrity


In [4]:
if 'is_fraud' not in df.columns:
    raise KeyError('Required target column is_fraud is missing.')

fraud_distribution_before = df['is_fraud'].value_counts(dropna=False).sort_index()
display(fraud_distribution_before.rename('before_cleaning'))

# Preserve the label: only convert safely to integer and remove rows where the label is missing.
df['is_fraud'] = pd.to_numeric(df['is_fraud'], errors='coerce')
missing_label_rows = int(df['is_fraud'].isna().sum())
df = df.dropna(subset=['is_fraud']).copy()
df['is_fraud'] = df['is_fraud'].astype(int)
print(f'Missing fraud labels removed: {missing_label_rows:,}')


is_fraud
0    10403
1      997
Name: before_cleaning, dtype: int64

Missing fraud labels removed: 0


## 5. Parse and standardize timestamps


In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
invalid_timestamp_count = int(df['timestamp'].isna().sum())
print(f'Invalid or missing timestamps: {invalid_timestamp_count:,}')

df = df.dropna(subset=['timestamp']).copy()
now_utc = pd.Timestamp.now(tz='UTC')
future_dated_count = int((df['timestamp'] > now_utc).sum())
print(f'Future-dated transactions removed: {future_dated_count:,}')
df = df.loc[df['timestamp'] <= now_utc].copy()


Invalid or missing timestamps: 61


Future-dated transactions removed: 0


## 6. Normalize text categories


In [6]:
text_columns = [
    'home_country', 'source_currency', 'dest_currency',
    'channel', 'ip_country', 'kyc_tier'
]

category_examples_before = {}
for column in ['channel', 'kyc_tier']:
    if column in df_raw.columns:
        category_examples_before[column] = df_raw[column].astype('string').value_counts(dropna=False).head(12)

for column in text_columns:
    if column in df.columns:
        df[column] = df[column].astype('string').str.strip().str.lower()


## 7. Fix inconsistent spellings


In [7]:
if 'channel' in df.columns:
    df['channel'] = (
        df['channel']
        .str.replace(r'\s+', ' ', regex=True)
        .replace({'moblie': 'mobile', 'mobille': 'mobile', 'weeb': 'web'})
    )

if 'kyc_tier' in df.columns:
    df['kyc_tier'] = df['kyc_tier'].replace({'standrd': 'standard', 'enhancd': 'enhanced'})

for column in ['home_country', 'source_currency', 'dest_currency', 'ip_country']:
    if column in df.columns:
        df[column] = df[column].str.upper()


## 8. Remove duplicates


In [8]:
duplicate_count_before = int(df_raw.duplicated().sum())
duplicates_after_timestamp_cleaning = int(df.duplicated().sum())
print(f'Duplicate rows before cleaning: {duplicate_count_before:,}')
print(f'Duplicate rows present before duplicate removal step: {duplicates_after_timestamp_cleaning:,}')

df = df.drop_duplicates().copy()
duplicate_count_after = int(df.duplicated().sum())
duplicates_removed = duplicates_after_timestamp_cleaning - duplicate_count_after
print(f'Duplicate rows after cleaning: {duplicate_count_after:,}')
print(f'Duplicates removed: {duplicates_removed:,}')


Duplicate rows before cleaning: 200
Duplicate rows present before duplicate removal step: 199
Duplicate rows after cleaning: 0
Duplicates removed: 199


## 9. Convert data types


In [9]:
def convert_bool_like(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype('boolean')
    mapping = {
        'true': True, 'false': False, '1': True, '0': False,
        'yes': True, 'no': False, 'y': True, 'n': False,
        't': True, 'f': False,
    }
    return series.astype('string').str.strip().str.lower().map(mapping).astype('boolean')

for column in ['new_device', 'location_mismatch']:
    if column in df.columns:
        df[column] = convert_bool_like(df[column])

numeric_columns = [
    'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest',
    'ip_risk_score', 'account_age_days', 'device_trust_score',
    'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h',
    'txn_velocity_24h', 'corridor_risk'
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

display(df.dtypes)


transaction_id                            object
customer_id                               object
timestamp                    datetime64[ns, UTC]
home_country                      string[python]
source_currency                   string[python]
dest_currency                     string[python]
channel                           string[python]
amount_src                               float64
amount_usd                               float64
fee                                      float64
exchange_rate_src_to_dest                float64
device_id                                 object
new_device                               boolean
ip_address                                object
ip_country                        string[python]
location_mismatch                        boolean
ip_risk_score                            float64
kyc_tier                          string[python]
account_age_days                           int64
device_trust_score                       float64
chargeback_history_c

## 10. Remove invalid transactions


In [10]:
non_negative_columns = numeric_columns
negative_mask = pd.Series(False, index=df.index)
negative_counts = {}

for column in non_negative_columns:
    if column in df.columns:
        column_negative_mask = df[column] < 0
        negative_counts[column] = int(column_negative_mask.sum())
        negative_mask = negative_mask | column_negative_mask.fillna(False)

invalid_negative_rows = int(negative_mask.sum())
print(f'Rows removed for impossible negative values: {invalid_negative_rows:,}')
display(pd.Series(negative_counts, name='negative_value_count'))

# Valid outliers are retained. Only impossible negative values in non-negative fields are removed.
df = df.loc[~negative_mask].copy()


Rows removed for impossible negative values: 200


amount_src                   100
amount_usd                     0
fee                          100
exchange_rate_src_to_dest      0
ip_risk_score                  0
account_age_days               0
device_trust_score           200
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h              200
txn_velocity_24h               0
corridor_risk                  0
Name: negative_value_count, dtype: int64

## 11. Missing values


In [11]:
missing_before = df_raw.isna().sum().rename('missing_before_cleaning')
display(missing_before[missing_before > 0].sort_values(ascending=False))

categorical_fill_columns = [
    'home_country', 'source_currency', 'dest_currency',
    'channel', 'ip_country', 'kyc_tier'
]
for column in categorical_fill_columns:
    if column in df.columns:
        missing_tokens = {'', 'unknown', 'nan', 'none', 'null', 'na', 'n/a', '<na>'}
        missing_like = df[column].astype('string').str.strip().str.lower().isin(missing_tokens)
        df.loc[missing_like, column] = pd.NA
        df[column] = df[column].fillna('Unknown')

# Numeric missing values are intentionally left for the modelling pipeline.
missing_after = df.isna().sum().rename('missing_after_cleaning')
display(missing_after[missing_after > 0].sort_values(ascending=False))


amount_usd            305
ip_address            305
ip_country            301
kyc_tier              300
fee                   295
device_trust_score    295
timestamp              29
Name: missing_before_cleaning, dtype: int64

amount_usd            290
fee                   290
ip_address            290
device_trust_score    290
Name: missing_after_cleaning, dtype: int64

## 12. Before-and-after illustrations


In [12]:
duplicate_illustration = pd.DataFrame({
    'step': ['Before duplicate removal', 'After duplicate removal'],
    'duplicate_rows': [duplicate_count_before, duplicate_count_after],
})
display(duplicate_illustration)


,step,duplicate_rows
0,Before duplicate removal,200
1,After duplicate removal,0


In [13]:
category_examples_after = {}
for column in ['channel', 'kyc_tier']:
    if column in df.columns:
        category_examples_after[column] = df[column].astype('string').value_counts(dropna=False).head(12)

for column in ['channel', 'kyc_tier']:
    if column in category_examples_before and column in category_examples_after:
        before = category_examples_before[column].rename('before_count')
        after = category_examples_after[column].rename('after_count')
        comparison = pd.concat([before, after], axis=1).fillna(0).astype(int)
        print(f'{column} before vs after standardisation')
        display(comparison)


channel before vs after standardisation


,before_count,after_count
channel,,
mobile,6366,6240
web,3727,3685
ATM,1008,0
mobille,60,0
mobile,48,0
MOBILE,47,0
unknown,37,0
WEB,36,0
web,34,0


kyc_tier before vs after standardisation


,before_count,after_count
kyc_tier,,
standard,7931,7786
enhanced,1829,1802
low,1039,1034
<NA>,300,0
standrd,72,0
STANDARD,70,0
standard,65,0
unknown,32,0
enhanced,17,0


## 13. Fraud distribution after cleaning and save output


In [14]:
fraud_distribution_after = df['is_fraud'].value_counts(dropna=False).sort_index()
display(pd.concat([
    fraud_distribution_before.rename('before_cleaning'),
    fraud_distribution_after.rename('after_cleaning')
], axis=1))

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Cleaned shape: {df.shape}')
print(f'Saved cleaned dataset to: {PROCESSED_PATH}')


,before_cleaning,after_cleaning
is_fraud,,
0,10403,9951
1,997,989


Cleaned shape: (10940, 26)
Saved cleaned dataset to: data\processed\cleaned_transactions.csv
